# 00. Train Model - Locally

Helper notebook for training fire risk model locally.

Recommended for making sure model + data are correct. Training full model locally is time consuming

IMPORTANT: Currently, multi-dir training shuffles features and labels so that they are mismatched
Need to fix either in geebeam export, in an offline post-processing step, or at training time by joining
based on index.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import keras
import aic_risk_modeling as arm
import matplotlib.pyplot as plt


In [ ]:
import json
with open('../configs/fusion_test_local.json', 'r') as f:
    config = json.load(f)

In [ ]:
# At its simplest: just run this
# arm.train.run(config)

In [ ]:
SEED = 54
RNG = np.random.default_rng(SEED)

# Set params

In [ ]:
DATA_DIRS = ['../../data/test_tf_data/2023/', '../../data/test_tf_data/2022','../../data/test_tf_data/2024']
TFRECORD_PATTERN='*.tfrecord.gz'

In [ ]:
# Merged dataset test, merging along features-axis
training_ds = arm.train.build_merged_dataset([DATA_DIRS[0], DATA_DIRS[1]], TFRECORD_PATTERN,  batch_size=4,  cache=False, axis='examples')

validation_ds = arm.train.data_loader.build_merged_dataset(
    [DATA_DIRS[2]],
    tfrecord_pattern='training-*.tfrecord.gz',
    batch_size=4,
    cache=False
    )

In [ ]:
# Normalization
normalize_list = arm.train.get_normalize_list(config)
norm_func = arm.train.create_normalizer(DATA_DIRS[1] + '/stats.pbtxt', normalize_list)
training_ds = training_ds.map(norm_func)
validation_ds = validation_ds.map(norm_func)

In [ ]:
# Select bands
training_ds = arm.train.data_loader.select_bands_transform(
    training_ds,
    input_feature_config=config['input_features'],
    output_feature_config=config['output_features'],

)
validation_ds = arm.train.data_loader.select_bands_transform(
    validation_ds,
    input_feature_config=config['input_features'],
    output_feature_config=config['output_features'],
)

In [ ]:
for inputs, labels in training_ds.take(1):
    print(labels.shape)
    print(inputs['im_annual'].shape)
    plt.imshow(inputs['im_annual'][0, 0, :, :, 0])
    plt.show()
    print(np.max(labels[0]))
    plt.imshow(labels[0])
    plt.show()

In [ ]:
all_models = arm.train.trainer.build_all_models(config['input_features'])

In [ ]:
model = arm.train.build_fusion(all_models)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0025),
    loss="Dice",
    metrics=[
        keras.metrics.BinaryIoU(target_class_ids=[1]),
        keras.metrics.AUC(),
    ]
    )

checkpoint_filepath = './checkpoint.model.keras'
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_loss',
    mode='min',
    save_best_only=True)

early_stopping_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=30)

model.fit(
    training_ds,
    validation_data=validation_ds,
    # class_weight={0:0.1, 1:0.9},
    epochs=25,
    callbacks=[model_checkpoint_callback, early_stopping_callback]
)

In [ ]:
new_model = tf.keras.models.load_model('checkpoint.model.keras')

In [ ]:
valid_masks = np.array([b[1][i].numpy() for b in training_ds for i in range(b[1].shape[0])])

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, jaccard_score

In [ ]:
out = new_model.predict(training_ds)

In [ ]:
valid_masks.max()

In [ ]:

print(f1_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(recall_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(precision_score(valid_masks.flatten()>0.5, out.flatten()>0.5))
print(jaccard_score(valid_masks.flatten()>0.5, out.flatten()>0.5))

In [ ]:
def visualize_risk_predict(input_batch, target_batch, output_i, batch_i, suptitle, cutoff=0.5):
    fig, axs = plt.subplots(1,3)
    fig.suptitle(suptitle)
    # Embeddings
    rgb = np.stack([
        input_batch['image'][batch_i, 2, :, :, 0].numpy(),
        input_batch['image'][batch_i, 2, :, :, 1].numpy(),
        input_batch['image'][batch_i, 2,:, :, 2].numpy()], axis=2)
    # shift
    vmin=-0.3
    vmax=0.3
    rgb = (rgb - vmin)/(vmax - vmin)
    axs.flatten()[0].imshow(rgb)
    axs.flatten()[0].set_title('Embeddings')


    # Prediction
    axs.flatten()[1].imshow(output_i)
    axs.flatten()[1].set_title('Predicted burned area 2024')

    # 2023 burn (target)
    axs.flatten()[2].imshow(target_batch[batch_i].numpy()>cutoff)
    axs.flatten()[2].set_title('Actual burned area 2024')
    fig.tight_layout()

    plt.show()


In [ ]:
import matplotlib.pyplot as plt
j = 0
for batch in training_ds:
    for i in range(batch[1].shape[0]):
        if (batch[1][i].numpy()>0.5).sum()>0 or (out[j]>0.5).sum()>0:
            visualize_risk_predict(
                input_batch = batch[0],
                target_batch = batch[1],
                output_i = out[j],
                batch_i=i,
                suptitle='Image {}'.format(j),
                cutoff=0.99
            )
        j+=1